# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

In [1]:
# Mount Google Drive to access datasets and save models
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [3]:
import chromadb

client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/chroma_db"
)

print(client.list_collections())

[Collection(name=sop_collection), Collection(name=langchain)]


In [4]:
import transformers
import peft
import torch

print(transformers.__version__)
print(peft.__version__)
print(torch.__version__)

5.13.1
0.19.1
2.11.0+cu128


In [5]:
#fixing compactability issues
!pip uninstall -y transformers peft accelerate

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0


In [6]:
#fixing compactability issues
!pip install -q \
transformers==4.55.0 \
peft==0.17.1 \
accelerate==1.10.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
import transformers
import peft
import torch

print(transformers.__version__)
print(peft.__version__)
print(torch.__version__)

4.55.0
0.17.1
2.11.0+cu128


### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [2]:
# YOUR CODE HERE
import json
import re
import torch
import chromadb

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

import pandas as pd

df_test = pd.read_csv(
    "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/df_test.csv"
)

print(df_test.shape)
print(df_test.columns)

(391, 9)
Index(['flags', 'instruction', 'category', 'intent', 'response',
       'chatml_instruction', 'input_ids', 'labels', 'attention_mask'],
      dtype='object')


In [3]:
# YOUR CODE HERE
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

LORA_PATH = "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/intent_lora_best"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Loading LoRA adapters...")

model = PeftModel.from_pretrained(
    base_model,
    LORA_PATH
)

print("Merging LoRA...")

model = model.merge_and_unload()

model.eval()

print("✓ Fine-tuned model ready.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading base model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading LoRA adapters...
Merging LoRA...
✓ Fine-tuned model ready.


In [4]:
client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/chroma_db"
)

collection = client.get_collection("langchain")

print("Documents in collection:", collection.count())

Documents in collection: 64


In [5]:
#router prompt
router_prompt = """
You are an intent classifier.

Return ONLY valid JSON.

Example 1
User: Where is my package?
{{"intent":"track_order"}}

Example 2
User: I want to cancel my order
{{"intent":"cancel_order"}}

Example 3
User: Update shipping address
{{"intent":"change_shipping_address"}}

Example 4
User: I want to place an order
{{"intent":"place_order"}}


User:
{query}
"""

In [24]:
#extract query
def extract_intent(query):

    prompt = router_prompt.format(query=query)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    decoded = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    # Extract JSON safely
    match = re.search(r"\{.*?\}", decoded, re.DOTALL)

    if match:

        try:

            parsed = json.loads(match.group())

            return parsed.get("intent", query)

        except:

            return query

    return query

In [25]:
#retrieve sop
def retrieve_sop(intent_text):

    results = collection.query(

        query_texts=[intent_text],

        n_results=1
    )

    if len(results["documents"][0]) == 0:

        return ""

    return results["documents"][0][0]

In [26]:
#RAG
def hybrid_rag(query):

    intent = extract_intent(query)
    sop = retrieve_sop(intent)
    prompt = f"""
You are a customer support assistant.
Answer the customer's question using ONLY the SOP below.
Do NOT repeat the SOP.
Do NOT output JSON.
Do NOT repeat the prompt.

SOP
----------------
{sop}

Customer Query:
{query}

Answer:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    if answer.strip().startswith("{"):
        answer = f"Based on our policy: {sop[:300].strip()}"

    return intent, sop, answer

In [27]:
print(next(model.parameters()).device)
print(model.hf_device_map if hasattr(model, "hf_device_map") else "No device map")

cuda:0
{'': 0}


In [28]:
test_query = df_test.iloc[0]["instruction"]

intent, sop, answer = hybrid_rag(test_query)

print("="*60)
print("Customer Query")
print("="*60)
print(test_query)

print("\n")

print("="*60)
print("Predicted Intent")
print("="*60)
print(intent)

print("\n")

print("="*60)
print("Retrieved SOP")
print("="*60)
print(sop[:600])

print("\n")

print("="*60)
print("Generated Answer")
print("="*60)
print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Customer Query
need help to shop several of ur item


Predicted Intent
add_to_cart


Retrieved SOP
# Payment Methods

## Accepted Methods
The store accepts major credit and debit cards, common digital wallets, and
bank transfer for eligible regions. Gift cards and store credit can be
combined with one other payment method at checkout. Cash on delivery is not
offered for online orders.


Generated Answer
Based on our policy: # Payment Methods

## Accepted Methods
The store accepts major credit and debit cards, common digital wallets, and
bank transfer for eligible regions. Gift cards and store credit can be
combined with one other payment method at checkout. Cash on delivery is not
offered for online orders.


### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [11]:
!pip install rouge-score sacrebleu rapidfuzz -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.9 MB/s eta 0:00:00


In [12]:
#imports
import re
import json
import pandas as pd

from rapidfuzz import fuzz
from rouge_score import rouge_scorer
import sacrebleu

In [29]:
#adversarial subset
pattern = r"still|never|terrible|frustrated"

adversarial_df = df_test[
    df_test["instruction"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )
]

print("Total test samples:", len(df_test))
print("Adversarial samples:", len(adversarial_df))

Total test samples: 391
Adversarial samples: 0


In [30]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

results = []

for _, row in df_test.iterrows():

    query = row["instruction"]
    true_intent = row["intent"]
    reference = row["response"]
    pred_intent, sop, answer = hybrid_rag(query)

    # -----------------------
    # Format adherence
    # -----------------------

    format_ok = isinstance(answer, str) and len(answer.strip()) > 0

    # -----------------------
    # Exact Match
    # -----------------------

    exact = pred_intent == true_intent

    # -----------------------
    # Fuzzy Match
    # -----------------------

    fuzzy = fuzz.ratio(
        pred_intent,
        true_intent
    ) >= 80

    # -----------------------
    # ROUGE
    # -----------------------

    rouge = scorer.score(
        reference,
        answer
    )

    rouge1 = rouge["rouge1"].fmeasure
    rougeL = rouge["rougeL"].fmeasure

    # -----------------------
    # BLEU
    # -----------------------

    bleu = sacrebleu.sentence_bleu(
        answer,
        [reference]
    ).score

    results.append({

        "instruction": query,
        "true_intent": true_intent,
        "predicted_intent": pred_intent,
        "format_ok": format_ok,
        "exact_match": exact,
        "fuzzy_match": fuzzy,
        "rouge1": rouge1,
        "rougeL": rougeL,
        "bleu": bleu,
        "answer": answer

    })


print("Evaluation completed.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

Evaluation completed.


### Evaluation on the `df_test` dataset

In [31]:
#metrics
results_df = pd.DataFrame(results)
format_score = results_df["format_ok"].mean() * 100
exact_score = results_df["exact_match"].mean() * 100
fuzzy_score = results_df["fuzzy_match"].mean() * 100
rouge1 = results_df["rouge1"].mean()
rougeL = results_df["rougeL"].mean()
bleu = results_df["bleu"].mean()

print("="*60)
print("Solution V2 Evaluation")
print("="*60)
print(f"Format Adherence : {format_score:.2f}%")
print(f"Exact Match      : {exact_score:.2f}%")
print(f"Fuzzy Match      : {fuzzy_score:.2f}%")
print(f"Average ROUGE-1  : {rouge1:.4f}")
print(f"Average ROUGE-L  : {rougeL:.4f}")
print(f"Average BLEU     : {bleu:.2f}")

Solution V2 Evaluation
Format Adherence : 100.00%
Exact Match      : 32.74%
Fuzzy Match      : 40.66%
Average ROUGE-1  : 0.2123
Average ROUGE-L  : 0.1333
Average BLEU     : 1.67


In [32]:
results_df[["instruction", "answer", "predicted_intent"]].head(10)

,instruction,answer,predicted_intent
0,need help to shop several of ur item,Based on our policy: # Payment Methods\n\n## A...,add_to_cart
1,i need assistance trying to update the shippin...,Based on our policy: ## Exceptions\nFor addres...,update_shipping_address
2,I do not know how to check the cancellation p...,"To help you with your query, I need to ask a c...",check_cancellation_fee
3,want assistance to see the payment modalities,Based on our policy: # Payment Methods\n\n## A...,payment
4,how can I get information about opening a stan...,"To help you open an account, we suggest creati...",create_account
5,I do not know how to see the status of order {...,Based on our policy: ## Order States\n- **Proc...,check_order_status
6,need help editing m address,I'm sorry to hear that you're having trouble w...,edit_shipping_address
7,how do I see at what time I can call customer ...,I'm sorry for the inconvenience you're experie...,contact_customer_service
8,using standard accouny,Based on our policy: # Password Reset\n\n## Sc...,account
9,I cannot afford order {{Order Number}},Based on our policy: ## Communication\nAcknowl...,cancel_order


### Evaluation on an Adversarial Subset

In [33]:
adv_results = []

for _, row in adversarial_df.iterrows():

    pred_intent, _, _ = hybrid_rag(row["instruction"])

    adv_results.append(

        fuzz.ratio(
            pred_intent,
            row["intent"]
        ) >= 80

    )

print("="*60)
print("Adversarial Evaluation")
print("="*60)
print(f"Samples : {len(adversarial_df)}")
if len(adversarial_df) == 0:
  print("No Adversarial samples found matching the pattern")
else:
  print(f"Accuracy : {100 * sum(adv_results)/len(adv_results):.2f}%")


Adversarial Evaluation
Samples : 0
No Adversarial samples found matching the pattern


In [34]:
OUTPUT_PATH = "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset"

results_df.to_csv(
    f"{OUTPUT_PATH}/v2_metrics.csv",
    index=False
)

print("✓ v2_metrics.csv saved")

✓ v2_metrics.csv saved


#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [35]:
import pandas as pd

# ===========================
# Load V1 metrics
# ===========================

v1 = pd.read_csv(
    "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/v1_metrics.csv"
)

# ===========================
# V2 metrics from Task 4.4.1
# ===========================

v2 = {
    "ROUGE-1": rouge1,
    "ROUGE-L": rougeL,
    "BLEU": bleu / 100,                # convert back to 0-1 scale
    "Consistency (%)": format_score,
    "Intent Accuracy (%)": exact_score
}

comparison = []

for _, row in v1.iterrows():

    metric = row["Metric"]
    naive = row["Naive RAG"]

    if metric not in v2:
        continue

    hybrid = v2[metric]

    improvement = (
        ((hybrid - naive) / naive) * 100
        if naive != 0 else 0
    )

    comparison.append({
        "Metric": metric,
        "Naive RAG": naive,
        "Hybrid RAG": hybrid,
        "% Improvement": improvement
    })

comparison_df = pd.DataFrame(comparison)

print("="*60)
print("Fine-Tuning Impact (V1 vs V2)")
print("="*60)
print(comparison_df)

comparison_df.to_csv(
    "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/FineTuning_Impact.csv",
    index=False
)

print("\n✓ FineTuning_Impact.csv saved.")

Fine-Tuning Impact (V1 vs V2)
            Metric  Naive RAG  Hybrid RAG  % Improvement
0          ROUGE-1     0.3434    0.212264     -38.187559
1          ROUGE-L     0.1775    0.133323     -24.888499
2             BLEU     0.0420    0.016724     -60.182118
3  Consistency (%)   100.0000  100.000000       0.000000

✓ FineTuning_Impact.csv saved.


In [36]:
import pandas as pd

v1_metrics = pd.read_csv(
    "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset/v1_metrics.csv"
)

print(v1_metrics.columns.tolist())
print(v1_metrics.head())

['Metric', 'Baseline', 'Naive RAG', '% Change']
                        Metric  Baseline  Naive RAG  % Change
0                      ROUGE-1    0.2801     0.3434   22.5976
1                      ROUGE-L    0.1532     0.1775   15.8165
2                         BLEU    0.0234     0.0420   79.5738
3              Consistency (%)  100.0000   100.0000    0.0000
4  Hallucination Frequency (%)   85.6777    74.1688   13.4328


### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [37]:
# YOUR CODE HERE
import os
import pandas as pd
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

OUTPUT_PATH = "/content/drive/MyDrive/upgrad AI/Assignment/RAG_Hybrid/Dataset/Dataset"

# ======================================================
# Helper: Baseline (no retrieval, raw query only)
# ======================================================
def baseline_response(query: str) -> str:
    prompt = f"Answer the following customer service question: {query}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# ======================================================
# Helper: Naive RAG (SOP retrieved by raw query, no router)
# ======================================================
def naive_rag(query: str) -> str:
    sop = retrieve_sop(query)
    prompt = f"""You are a customer support assistant.
Answer the customer's question using ONLY the SOP below.

SOP
---
{sop}

Customer Query:
{query}

Answer;"""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# ======================================================
# Comparative Evaluation: Baseline vs Naive RAG vs Hybrid RAG
# Hybrid RAG results are reused from Task 4.4.1 (cell-23)
# to avoid re-running the 15-min evaluation loop.
# ======================================================

comp_scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

smooth = SmoothingFunction().method1

comp_results = []

print("="*60)
print("Running Comparative Evaluation")
print("="*60)

evaluation_df = df_test

for _, row in tqdm(evaluation_df.iterrows(), total=len(evaluation_df)):

    query = row["instruction"]
    reference = row["response"]

    # ==========================================
    # Baseline
    # ==========================================
    try:
        baseline = baseline_response(query)
    except:
        baseline = ""

    # ==========================================
    # Naive RAG
    # ==========================================
    try:
        naive = naive_rag(query)
    except:
        naive = ""

    # ==========================================
    # Hybrid RAG - Reuse from Task 4.4.1 results
    # ==========================================
    try:
        hybrid = results_df.loc[results_df["instruction"] == query, "answer"].values[0]
    except:
        hybrid = ""

    # ==========================================
    # Metrics helper
    # ==========================================
    def compute_scores(pred):
        rouge = comp_scorer.score(reference, pred)

        bleu = sentence_bleu(
            [reference.split()],
            pred.split(),
            smoothing_function=smooth
        )

        return (
            rouge["rouge1"].fmeasure,
            rouge["rougeL"].fmeasure,
            bleu
        )

    b_r1, b_rl, b_bleu = compute_scores(baseline)
    n_r1, n_rl, n_bleu = compute_scores(naive)
    h_r1, h_rl, h_bleu = compute_scores(hybrid)

    comp_results.append({
        "instruction": query,
        "Baseline_ROUGE1": b_r1,
        "Baseline_ROUGEL": b_rl,
        "Baseline_BLEU": b_bleu,

        "Naive_ROUGE1": n_r1,
        "Naive_ROUGEL": n_rl,
        "Naive_BLEU": n_bleu,

        "Hybrid_ROUGE1": h_r1,
        "Hybrid_ROUGEL": h_rl,
        "Hybrid_BLEU": h_bleu,

        "Baseline_Output": baseline,
        "Naive_Output": naive,
        "Hybrid_Output": hybrid,
        "Reference": reference
    })


# ======================================================
# Save Full Results
# ======================================================
comp_results_df = pd.DataFrame(comp_results)

full_path = os.path.join(
    OUTPUT_PATH,
    "Comparative_Results_Full.csv"
)

comp_results_df.to_csv(full_path, index=False)


# ======================================================
# Summary Table
# ======================================================
summary = pd.DataFrame({
    "Architecture": [
        "Baseline",
        "Naive RAG",
        "Hybrid RAG"
    ],

    "ROUGE1": [
        comp_results_df["Baseline_ROUGE1"].mean(),
        comp_results_df["Naive_ROUGE1"].mean(),
        comp_results_df["Hybrid_ROUGE1"].mean()
    ],

    "ROUGEL": [
        comp_results_df["Baseline_ROUGEL"].mean(),
        comp_results_df["Naive_ROUGEL"].mean(),
        comp_results_df["Hybrid_ROUGEL"].mean()
    ],

    "BLEU": [
        comp_results_df["Baseline_BLEU"].mean(),
        comp_results_df["Naive_BLEU"].mean(),
        comp_results_df["Hybrid_BLEU"].mean()
    ]
})

summary_path = os.path.join(
    OUTPUT_PATH,
    "Comparative_Results_Summary.csv"
)

summary.to_csv(summary_path, index=False)

print("\nComparative_Results_Full.csv saved")
print("Comparative_Results_Summary.csv saved")

display(summary)

Running Comparative Evaluation


  0%|          | 0/391 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'


Comparative_Results_Full.csv saved
Comparative_Results_Summary.csv saved


,Architecture,ROUGE1,ROUGEL,BLEU
0,Baseline,0.226555,0.136359,0.012004
1,Naive RAG,0.211928,0.136588,0.007969
2,Hybrid RAG,0.212264,0.133323,0.007890


---
## END-OF-NOTEBOOK CHECKLIST (FINAL)

> **IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [ ] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [ ] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [ ] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [ ] **4.5** All 3 architectures scored with SOP-grounded references
- [ ] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [ ] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**